# EGM 722 Project Notebook: Greenspace Analysis (Draft)

## Introduction

Welcome to my project notebook! This will take you through the code required to perform analysis on the amount of Greenspace available in Northern Ireland. 

The code will consist of 4 main parts:
1. **Importing the necessary modules**
2. **Loading the data and setting the CRS**
3. **Carrying out the analysis**
4. **Mapping the results**

**Please Note:** This is currently only a draft - parts may be messy or incomplete!

## Part 1: Preparing Data

To start the project, we first need to import the necessary modules, followed by the data.

In [1]:
#Import the required modules
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from cartopy.feature import ShapelyFeature
import cartopy.crs as ccrs

In [2]:
# Load and name the data layers
greenspace = gpd.read_file(os.path.abspath('data_files/Greenspace_Phase2_06052022.shp'))
lgd2014 = gpd.read_file(os.path.abspath('data_files/LGD2014.shp'))
dea2014 = gpd.read_file(os.path.abspath('data_files/dea2014.shp'))
dz2021 = gpd.read_file(os.path.abspath('data_files/DZ2021.shp'))
sdz2021 = gpd.read_file(os.path.abspath('data_files/SDZ2021.shp'))


In [63]:
# View the first 5 rows of the greenspace data to check if it loaded in correctly
sdz2021

,SDZ2021_cd,SDZ2021_nm,DEA2014_cd,DEA2014_nm,LGD2014_cd,LGD2014_nm,Area_ha,Perim_km,geometry,sdz_area
0,N21000001,Dunsilly_A,N10000104,Dunsilly,N09000001,Antrim and Newtownabbey,6611.41,57.65,"POLYGON ((302828.621 393821.676, 302831.93 393...",66.114052
1,N21000002,Dunsilly_B,N10000104,Dunsilly,N09000001,Antrim and Newtownabbey,7296.19,51.27,"POLYGON ((319895.281 396949.987, 319893.771 39...",72.961893
2,N21000003,Dunsilly_C,N10000104,Dunsilly,N09000001,Antrim and Newtownabbey,2794.37,28.18,"POLYGON ((302824.752 393819.021, 302828.603 39...",27.943745
3,N21000004,Dunsilly_D,N10000104,Dunsilly,N09000001,Antrim and Newtownabbey,111.61,7.45,"POLYGON ((307690.541 389508.86, 307599.464 389...",1.116141
4,N21000005,Dunsilly_E,N10000104,Dunsilly,N09000001,Antrim and Newtownabbey,80.94,5.43,"POLYGON ((308921.837 390709.973, 308934.877 39...",0.809400
...,...,...,...,...,...,...,...,...,...,...
845,N21000846,Ards_Peninsula_H,N10001101,Ards Peninsula,N09000011,Ards and North Down,2961.58,44.52,"POLYGON ((365372.33 358780.347, 365391.438 358...",29.615817
846,N21000847,Ards_Peninsula_J,N10001101,Ards Peninsula,N09000011,Ards and North Down,118.41,7.82,"POLYGON ((365372.33 358780.347, 365372.362 358...",1.184061
847,N21000848,Ards_Peninsula_K,N10001101,Ards Peninsula,N09000011,Ards and North Down,89.58,9.82,"POLYGON ((365256.349 358088.025, 365257.143 35...",0.895834
848,N21000849,Ards_Peninsula_L,N10001101,Ards Peninsula,N09000011,Ards and North Down,5958.44,74.01,"POLYGON ((362545.425 345473.599, 362547.132 34...",59.584396


Now that the data is loaded and looks okay, we will check the crs of each layer, and make any necessary ammendments

In [4]:
# Check the crs of each layer
layer_crs = {'greenspace_crs': [greenspace.crs], 'lgd2014_crs': [lgd2014.crs], 'dz2021_crs': [dz2021.crs], 'sdz2021_crs': [sdz2021.crs]}
crs_table = pd.DataFrame(data=layer_crs)
crs_table

,greenspace_crs,lgd2014_crs,dz2021_crs,sdz2021_crs
0,EPSG:29902,EPSG:29902,EPSG:29902,EPSG:29902


In [5]:
ni_utm = ccrs.UTM(29) # create a Universal Transverse Mercator reference system to transform our data.

In [6]:
ccrs.CRS(greenspace.crs) # create a cartopy CRS representation of the CRS associated with the outline dataset


<Projected CRS: EPSG:29902>
Name: TM65 / Irish Grid
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Ireland - onshore.
- bounds: (-10.56, 51.39, -5.93, 55.43)
Coordinate Operation:
- name: Irish Grid
- method: Transverse Mercator
Datum: TM65
- Ellipsoid: Airy Modified 1849
- Prime Meridian: Greenwich

You may have noticed that the greenspace polygon has multiple 'Polgon Z' and 'Multipolygon Z' Types we want rid of these.

In [7]:
greenspace_sp=greenspace.explode(column=None)

## Part 2: Greenspace Analysis

This next part of the notebook will demonstrate some analysis using the greenspace data and boundaries provided.

Using the data, we can find:
1. What areas have the highest/lowest coverage of greenspace: at SDZ level and DEA, and per 1000 people.
2. What areas (SDZ) are closest or within a certain distance of greenspaces
3. Bonus: How many parcels are available within each DEA that are available for greenspace? These must be over a certain size, be a natural landscape, within distance of roads, within distance of houses and not already within a greenspace. Which DEAs have the most of these?

So, lets begin...


### Calculating coverage

Before we can calculate what area of greenspace is found within each SDZ, LGD, DEA and so on. It would be useful to calculate the area of each polygon to a standard unit.
This is where our first fucntion will be introduced, one that calculates the area of each polygon, in Km2 and adds it on as a new column to the table.

In [8]:
def area_calc(layer, col_name):
    """Caluclates the area of each polygon in the layer and returns a total.
    
    Parameters 
    layer : input polygon layer
    col_name : name of the area column

    Returns : sum_area_SQKM
        Prints the total area of the polygons in squared kilometers
    """

    layer[col_name] = layer['geometry'].area/1e6

    sum_area_SQKM = layer[col_name].sum()
    print(f'The total area is {sum_area_SQKM:.2f} kilometers squared')

In [9]:
#test area_calc function on greenspace layer
area_calc(greenspace_sp, 'greenspace_area_SQKM')

The total area is 875.94 kilometers squared


In [10]:
#test area_calc function column on greenspace layer
greenspace_sp

,SourceID,GUID,Name,Source,Category,Type,PaidAccess,Area_Ha,Verified,ShowOnMap,ORNI_ID,DataAdded,SiteCreate,geometry,greenspace_area_SQKM
0,NaN,68d2df20-1512-4218-a759-1000732e93b0,Conservation Volunteers NI,LPS and Outscape,Amenity Greenspace,Community Garden,No,0.668234,Approximated,Yes,51,2022-09-21,Pre 2023,"POLYGON Z ((335088.596 367463.239 0, 334995.26...",0.006682
1,NaN,7b20f220-682e-4566-a4fe-2513cc65ed6e,Lough Shore Park,Antrim and Newtownabbey Borough Council,Parks and Gardens,Public Park,No,6.598284,Approximated,Yes,61,2022-09-21,Pre 2023,"POLYGON Z ((313511.357 386601.644 0, 313465.51...",0.065983
2,NaN,8a4b8a31-eba9-4e7a-b8d5-cea0724d848d,Ardmore Recreation Centre,"Armagh City, Banbridge and Craigavon Borough C...",Amenity Greenspace,Playing Fields,No,2.969549,Approximated,Yes,67,2022-09-21,Pre 2023,"POLYGON Z ((289345.456 344248.549 0, 289352.83...",0.029695
3,NaN,36f2a040-649c-4eb5-9174-7b25c083f489,Clare Glen - Phase 3 - Link Footpath/River Wal...,"Armagh City, Banbridge and Craigavon Borough C...",Woodland,Woodland,No,3.396109,Approximated,Yes,69,2022-09-21,Pre 2023,"POLYGON Z ((303292.783 345625.097 0, 303291.38...",0.033961
4,NaN,2f586e46-b9ee-4d13-83b1-9b254fa3a698,"Folly Glen, Armagh","Armagh City, Banbridge and Craigavon Borough C...",Woodland,Woodland,No,7.849400,Approximated,Yes,70,2022-09-21,Pre 2023,"POLYGON Z ((288701.465 343901.121 0, 288688.70...",0.002338
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1487,NaN,NaN,Twinbrook Wildlife Park,Belfast City Council,Amenity Greenspace,Public Open Space,NaN,1.161658,NaN,Yes,0,1899-12-30,NaN,"POLYGON Z ((328479.994 368987.968 0, 328473.32...",0.010994
1487,NaN,NaN,Twinbrook Wildlife Park,Belfast City Council,Amenity Greenspace,Public Open Space,NaN,1.161658,NaN,Yes,0,1899-12-30,NaN,"POLYGON Z ((328487.187 369000.669 0, 328484.11...",0.000622
1488,NaN,NaN,Drumcoo Playing Fields,Mid Ulster District Council,Amenity Greenspace,Playing Fields,NaN,3.022465,NaN,Yes,0,1899-12-30,Pre 2023,"POLYGON Z ((279812.935 363921.041 0, 279810.78...",0.030225
1489,NaN,NaN,Battery Harbour Park,Mid Ulster District Council,Amentiy Greenspace,Public Open Space,NaN,1.769502,NaN,Yes,0,1899-12-30,NaN,"POLYGON Z ((296502.851 377054.243 0, 296514.15...",0.017695


In [11]:
help(area_calc)

Help on function area_calc in module __main__:

area_calc(layer, col_name)
    Caluclates the area of each polygon in the layer and returns a total.

    Parameters
    layer : input polygon layer
    col_name : name of the area column

    Returns : sum_area_SQKM
        Prints the total area of the polygons in squared kilometers



In [12]:
#add a area SQKM column to the remainder of the datasets and find their total area
area_calc(dz2021, 'dz_area')
area_calc(sdz2021, 'sdz_area')
area_calc(dea2014, 'dea_area')
area_calc(lgd2014, 'lgd_area')

The total area is 13628.31 kilometers squared
The total area is 13628.31 kilometers squared
The total area is 14315.28 kilometers squared
The total area is 14315.28 kilometers squared


Now we are ready to start analysing.


### Trying a new method: clipping!

In [69]:
def coverage_calc(layer, clipping_feature, orig_area):
    """Clips the selected greensapce layer to the selected dataset, creates a grouped gdf based on the individual features, which is then joined to the original selected dataset to calculate the % coverage

    Parameters: 
    layer - the selected layer or dataset to be clipped to
    clipping feature - the column name of the clipping dataset, must be a unique
    oirg_area - the calculated area column in the clipping dataset

    Returns:
    Output table - joined table of the original layer with information on its total coverage of greenspace
    """
    if layer[clipping_feature].is_unique == False:
        raise Exception('Clipping features must be unique.')
    
    clipped = []
    for selected_areas in layer[clipping_feature]:
        tmp_clip = gpd.clip(greenspace_sp, layer[layer[clipping_feature] == selected_areas])
        tmp_clip['greenspace_area_SQKM'] = tmp_clip['geometry'].area/1e6
        tmp_clip[clipping_feature] = selected_areas
    
        clipped.append(tmp_clip)

    clipped_gdf = gpd.GeoDataFrame(pd.concat(clipped, ignore_index=True))
    
    grouped_gdf = pd.DataFrame(index=layer[clipping_feature]) # Creates a grouped dataframe series that sums the total area of greenspace in the clipped layer
    grouped_gdf['greenspace_coverage_SQKM'] = clipped_gdf.groupby([clipping_feature])['greenspace_area_SQKM'].sum()

    output_table = pd.merge(layer, grouped_gdf, on = clipping_feature) # Joins the grouped data frame to the original dataset, and calculated the % coverage
    output_table['pc_coverage'] = output_table['greenspace_coverage_SQKM'] / output_table[orig_area] * 100

    return output_table

In [ ]:
sdz_coverage = coverage_calc(sdz2021, 'SDZ2021_nm', 'sdz_area')

In [73]:
sdz_coverage.groupby(['DEA'])['pc_coverage'].sum().sort_values(ascending=False)

NameError: name 'sdz_coverage' is not defined

In [49]:
lgd_coverage = coverage_calc(lgd2014, 'LGDNAME', 'lgd_area')

In [50]:
lgd_coverage

,OBJECTID,LGDNAME,LGDCode,popn,Shape_Leng,Shape_Area,geometry,lgd_area,greenspace_coverage_SQKM,pc_coverage
0,1.0,Antrim and Newtownabbey,N09000001,141428,165113.558771,7.278895e+08,"POLYGON ((319598.922 397397.797, 319595.68 397...",727.889478,12.425218,1.707020
1,2.0,Armagh City Banbridge and Craigavon,N09000002,209276,242967.696647,1.435980e+09,"POLYGON ((311885.478 366349.627, 311904.219 36...",1435.980451,29.361744,2.044718
2,3.0,Belfast,N09000003,337885,65905.220983,1.377021e+08,"POLYGON ((334071.541 379980.088, 334096.675 37...",137.702072,20.517916,14.900223
3,4.0,Causeway Coast and Glens,N09000004,143541,381602.781803,1.983852e+09,"MULTIPOLYGON (((297026.984 446000.593, 297033....",1983.851819,198.972460,10.029603
4,5.0,Derry City and Strabane,N09000005,150134,271234.988347,1.248976e+09,"POLYGON ((247533.964 424601.021, 247536.214 42...",1248.976336,76.646643,6.136757
5,6.0,Fermanagh and Omagh,N09000006,115794,426211.823548,3.005966e+09,"POLYGON ((266554.895 392093.167, 266557.237 39...",3005.965778,286.400462,9.527735
6,7.0,Lisburn and Castlereagh,N09000007,142587,167365.431586,5.103119e+08,"POLYGON ((343158.45 376583.005, 343162.985 376...",510.311902,5.642238,1.105645
7,8.0,Mid and East Antrim,N09000008,137427,234832.258324,1.061065e+09,"POLYGON ((329896.206 424716.479, 329900.103 42...",1061.065214,30.279666,2.853705
8,9.0,Mid Ulster,N09000009,145391,318235.783575,1.955057e+09,"POLYGON ((294150.193 412224.918, 294156.626 41...",1955.056614,73.207589,3.744525
9,10.0,Newry Mourne and Down,N09000010,178794,380041.040234,1.681856e+09,"POLYGON ((341276.618 362805.051, 341283.866 36...",1681.856446,117.673445,6.996640


SyntaxError: invalid syntax (1129111765.py, line 1)